In [3]:
import numpy as np
import jax.numpy as jnp
import jax

In [4]:
from jax import jit
import optax

key = jax.random.PRNGKey(0)

def grads(
    zi, weights, V, X,Z):
    penalty_grads = -(zi.T @ Z * V.T)@ weights
    grads = X.T @ zi + penalty_grads
    return grads / X.shape[0]

def update_with_grads(vi, grads, opt_state):
    updates, opt_state = optimizer.update(-grads, opt_state)
    vi_new = optax.apply_updates(vi, updates)
    vi_new /= jnp.linalg.norm(vi_new)
    return vi_new, opt_state

v_grads = jax.vmap(
                grads,
                in_axes=(0, 0, None,None, None),
            )

def update(X,V,weights,opt_state):
    Zx=X@V.T
    grads_v = v_grads(Zx.T,weights, V, X,Zx)
    V, opt_state = update_with_grads(
        V, grads_v, opt_state
    )
    return V, opt_state

update_jit = jax.jit(update)

In [5]:
n_components=4
p=784
weights = jnp.ones((n_components, n_components)) - jnp.eye(
            n_components
        )
weights = weights.at[jnp.triu_indices(n_components, 1)].set(0)
V=jax.random.normal(key, (n_components,p))
optimizer = optax.sgd(learning_rate=1e-3)
opt_state = optimizer.init(V)

In [6]:
import array
import gzip
import os
import struct
import urllib.request
from os import path
import jax.numpy as jnp
import numpy as np

from ccagame.utils import data_stream
from cca_zoo.models import rCCA

_DATA = "/tmp/jax_example_data/"


def _download(url, filename):
    """Download a url to a file in the JAX data temp directory."""
    if not path.exists(_DATA):
        os.makedirs(_DATA)
    out_file = path.join(_DATA, filename)
    if not path.isfile(out_file):
        urllib.request.urlretrieve(url, out_file)
        print("downloaded {} to {}".format(url, _DATA))


def _partial_flatten(x):
    """Flatten all but the first dimension of an ndarray."""
    return np.reshape(x, (x.shape[0], -1))


def _one_hot(x, k, dtype=np.float32):
    """Create a one-hot encoding of x of size k."""
    return np.array(x[:, None] == np.arange(k), dtype)


def mnist_raw():
    """Download and parse the raw MNIST dataset."""
    # CVDF mirror of http://yann.lecun.com/exdb/mnist/
    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    def parse_labels(filename):
        with gzip.open(filename, "rb") as fh:
            _ = struct.unpack(">II", fh.read(8))
            return np.array(array.array("B", fh.read()), dtype=np.uint8)

    def parse_images(filename):
        with gzip.open(filename, "rb") as fh:
            _, num_data, rows, cols = struct.unpack(">IIII", fh.read(16))
            return np.array(array.array("B", fh.read()), dtype=np.uint8).reshape(
                num_data, rows, cols
            )

    for filename in [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz",
    ]:
        _download(base_url + filename, filename)

    train_images = parse_images(path.join(_DATA, "train-images-idx3-ubyte.gz"))
    train_labels = parse_labels(path.join(_DATA, "train-labels-idx1-ubyte.gz"))
    test_images = parse_images(path.join(_DATA, "t10k-images-idx3-ubyte.gz"))
    test_labels = parse_labels(path.join(_DATA, "t10k-labels-idx1-ubyte.gz"))

    return train_images, train_labels, test_images, test_labels


def mnist(permute_train=False):
    """Download, parse and process MNIST data to unit scale and one-hot labels."""
    train_images, train_labels, test_images, test_labels = mnist_raw()

    train_images = _partial_flatten(train_images) / np.float32(255.0)
    test_images = _partial_flatten(test_images) / np.float32(255.0)
    train_labels = _one_hot(train_labels, 10)
    test_labels = _one_hot(test_labels, 10)

    if permute_train:
        perm = np.random.RandomState(0).permutation(train_images.shape[0])
        train_images = train_images[perm]
        train_labels = train_labels[perm]

    return train_images, train_labels, test_images, test_labels


def mnist_iterator(batch_size, n_components, pca=False, cca=False, p=392):
    X, _, X_te, _ = mnist()
    X += np.random.normal(size=X.shape)
    X_te += np.random.normal(size=X_te.shape)
    if pca:
        correct_U, _, _ = np.linalg.svd(X.T @ X)
        correct_U = correct_U[:, :n_components]
        return data_stream(X, batch_size=batch_size), X_te, correct_U, X.shape[1]
    else:
        Y = X[:, p:]
        X = X[:, :p]
        Y_te = X_te[:, p:]
        X_te = X_te[:, :p]
        if cca:
            cca = rCCA(latent_dims=n_components, scale=False, centre=False, c=0.01).fit(
                (X, Y)
            )
            correct_U, correct_V = cca.weights
            correct_U /= np.linalg.norm(correct_U, axis=0)
            correct_V /= np.linalg.norm(correct_V, axis=0)
        else:
            correct_U, _, correct_V = np.linalg.svd(X.T @ Y)
            correct_U = correct_U[:, :n_components]
            correct_V = correct_V[:n_components, :].T  
        return (
            data_stream(X, Y=Y, batch_size=batch_size),
            (X_te, Y_te),
            (correct_U, correct_V),
            (X.shape[1], Y.shape[1]),
        )

ModuleNotFoundError: No module named 'jaxline'

In [ ]:
data,holdout,correct_eigenvectors,dims= mnist_iterator(
                batch_size=1024, n_components=n_components, pca=True)

NameError: name 'mnist_iterator' is not defined

In [ ]:
for i in range(10000):
    x=next(data)
    V, opt_state=update_jit(x,V,weights,opt_state)

NameError: name 'data' is not defined